In [ ]:
# QubiC toolchain for compiling circuits
import qubic.toolchain as tc

# QubiC configuration management libraries
import qubitconfig.qchip as qc
from distproc.hwconfig import FPGAConfig, load_channel_configs

# other packages
import numpy as np
import matplotlib.pyplot as plt

# Outline
1. Play a simple pulse:
   - introduction to QubiC pulse programming
   - overview of QubiC compiler tools
   - connect to simulator and view simulated outputs
2. Multiple pulses + timing constraints: delay and barrier
3. Introduction to quantum logic gates
   - simulations with QuTiP
5. Overview of Readout Signal Chain
   - play pulses on multiplexed readout drive
   - readout demodulation + integration
6. Mid-circuit measurement based feedforward
7. Loops, variables, and ALU operations
   - construct frequency, phase, and amplitude parameter sweeps

# Hardware (and Simulator) Channel Configuration

At the lowest level, any quantum computation consists of a series of analog control and measurement pulses applied to the physical qubits (and/or readout resonators). QubiC 2.0 uses the [distributed processor](https://gitlab.com/LBL-QubiC/distributed_processor) for configuring and triggering pulses. In the current configuration, each processor core is connected to three process element (signal generator) channels in the firmware: **qubit drive, readout drive, and readout demodulation**:
![image](images/single_core_config.png)

Each drive channel connects to its own DAC, while the readout drive (demod) are multiplexed to a single DAC (ADC). So the full architecture looks like this:

![image](images/channel_diagram.drawio.svg)

**NOTE**: We don't have physical DACs for this part of the tutorial, so simulated DAC drive channels are indexed by `core_ind`; i.e. `core_0, elem_0 --> dac_out[0]`, `core_1, elem_0 --> dac_out[1]`, etc.

The core --> signal generator mapping as well as the ADC/DAC multiplexing configuration is fully configurable at synthesis time; see the [gateware](https://gitlab.com/LBL-QubiC/gateware) repo for details. For example, each drive DAC can be configured to be the sum of two different sig gen outputs, each connected to a different core.

# Pulse Nomenclature

<table style="padding: 0px"><tr></tr><tr>
<td style="width: 50%; padding: 0px">
<div align="left" style="text-align: left; font-size: 120%">
<p>A <i>pulse</i> is series of output voltages at specific time stamps.
It is constructed by first defining a pulse <i>envelope</i> at regular steps called <i>samples</i>.
For example, an envelope could take the form of a Gaussian shape, with at each sample the value corresponding to a Gaussian function.
Samples become time stamps by specifying to QubiC the <i>sample rate</i> and <i>interpolation ratio</i> to get the number of samples played per second.</p>
<p>The qubit oscillates, however, so to ensure that it actually "sees" the envelope as intended, the signal needs to be modulated with a <i>carrier signal</i> to transform in from the laboratory frame of reference into the qubit one.
Finally, the scale of the pulse is determined by specifying a pulse <i>amplitude</i>.</p>
</div></td>
<td style="width: 50%">
  <img src='./images/Illustration_of_Amplitude_Modulation.png'>
</td>
</tr>
<tr></tr><tr><td></td>
<td><div style="font-size: 70%">Source: Wikipedia</div></td>
</table>
A pulse envelope can be constructed directly and provided to QubiC in the form of a <span style="font-family:monospace">numpy</span> array, exactly specifying the output value for each sample.
However, it is much more efficient to provide parametrizations of pre-defined envelopes, such as those from QubiC's <i>pulse library</i> of commonly used ones.
Such a parametrized envelope can be reused, e.g. different qubits can share the same envelope shape but different carrier frequencies and amplitudes, thus reducing the overall memory requirements of the system, which is essential to achieve scale.


# Play a simple pulse

## Load configuration information

1) define FPGA config; this has **timing information** for the scheduler as well as **function processor** (FPROC) **channel information**. This config only changes with the **gateware**; for the current version it is fine to use the following default config:

In [ ]:
fpga_config = FPGAConfig()

2. load channel configs, which assign **named output channels** to **core** and **elem** indices and configure signal generator parameters. These **core + elem indices correspond to physical output channels** (or readout downconversion channels); see the channel map diagram above, or [Understanding Channel Configuration](https://gitlab.com/LBL-QubiC/software/-/wikis/Understanding-Channel-Configuration) for more details. This configuration only changes with **gateware** and **physical wiring**.


In [ ]:
channel_configs = load_channel_configs('./config/channel_config.json')

example channel config for named output 'Q0.qdrv' (qubit drive channel for physical qubit 'Q0'): 

```json
"Q0.qdrv":{
        "core_ind": 7,
        "core_name": "qubit",
        "elem_ind": 0,
        "elem_type": "rf",
        "elem_params":{
            "samples_per_clk": 16,
            "interp_ratio": 1
        },
        "env_mem_name": "qdrvenv{core_ind}",
        "freq_mem_name": "qdrvfreq{core_ind}",
        "acc_mem_name": "accbuf{core_ind}"
}
```

The physical output channel is given by `core_ind` and `elem_ind`, while signal generator params are given by `elem_params`. Note that `elem_params` for a given signal generator are fixed and can only be changed by modifying the gateware.

## Define the pulse sequence

A QubiC circuit (or "program") is represented natively using [Qubic-IR](https://lbl-qubic.gitlab.io/distributed_processor), formatted as a **list of dictionaries**. Each dictionary represents a single instruction, which could be a **pulse command, program statement, or timing construct**.

In [ ]:
circuit = [
    #play a single pulse on the Q0 drive channel
    {'name': 'pulse', 'phase': 0, 'freq': 4944383311, 'amp': 0.334704954261188, 'twidth': 2.4e-08,
     'env': {'env_func': 'cos_edge_square', 'paradict': {'ramp_fraction': 0.25}},
     'dest': 'Q0.qdrv'}, 
    
]

The pulse-dictionary above includes both parameters for the carrier signal (**phase, freq, and amp**), as well as a reference to a function that provides the **envelope** used for **amplitude modulation**. See [qubitconfig.envelope_pulse](https://gitlab.com/LBL-QubiC/experiments/qubitconfig/-/blob/main/qubitconfig/envelope_pulse.py?ref_type=heads) to see all available envelope functions. Envelopes are computed at compile-time, so a user could add arbitrary custom envelope functions to this library. Alternatively, a **numpy array of samples** can be provided. 

![image](images/pulse_example.svg)

## Compile and Assemble

Compile the program. Since we don't have any references to gates, we can pass None to the qchip argument. Broadly, the compiler does the following:
 1. Resolve Gate references into pulses *(don't worry about this for now, we'll get to this later)*
 2. Schedule all pulses
 3. Lower control flow *(also don't worry about this)*
 4. Break up the program into a set of several programs, one per core

The output of the compile stage is a distributed processor assembly program, which consists of initialization/termination statements, as well as a list of scheduled pulses for each core. Our program only uses one processor core, scoped to channels `('Q0.qdrv', 'Q0.rdrv', 'Q0.rdlo')`.

### Compiler Flags:

The CompilerFlags object (or equivalent dictionary) can be used to configure which IR passes are run during compilation. To see the available options, load the class:'

In [ ]:
from distproc.compiler import CompilerFlags
# todo: fix compiler flag docs
CompilerFlags?

In [ ]:
# CompilerFlags can be set using an instance or dictionary. Here, we set resolve_gates to false since we 
# are programming at the pulse level only.
compiled_prog = tc.run_compile_stage(circuit, fpga_config, None, compiler_flags={'resolve_gates': False})
compiled_prog = tc.run_compile_stage(circuit, fpga_config, None, compiler_flags=CompilerFlags(resolve_gates=False))
compiled_prog.program

Run the assembler to convert the above program into machine code that we can load onto the FPGA or gateware simulation:

In [ ]:
asm_prog = tc.run_assemble_stage(compiled_prog, channel_configs)

## Instantiate the Simulator and Run Program

The QubiC emulator is a python program that can run compiled QubiC binaries and simulate the resulting physical DAC output signals (as well as readout demodulation signals). It provides a **clock cycle accurate** simulation of the QubiC distributed core and signal generation/demod chain. Below, we instantiate the simulator, and inspect the resulting behavior of the DAC output.

In [ ]:
from emulator.emulator import Emulator

Emulator.generate_hwcfg("./config/hw_config.json", channel_configs, "./config/dsp_config.yaml", FPGAConfig())
Emulator.generate_qubitcfg("./config/em_qubit_config.json", "./config/qubitcfg.json")

em = Emulator(chanconfig_path='./config/channel_config.json', hwconfig_path='./config/hw_config.json', qubitconfig_path='./config/em_qubit_config.json')

In [ ]:
# send our program to the simulator, and run it for 150 ns

# Load the assembled binaries into the emulator, which will parse the command, envelope, and frequency buffers
em.load_program(asm_prog)
result = em.execute()

The DAC output is stored in `client.dac_out`. We expect to see our pulse on **DAC channel 7**, since `channel_configs['Q0.qdrv'].core_ind = 7`. 

In [ ]:
result.graph_channel('Q0.qdrv')

We can use the client object to plot this output:

# Multiple Pulses

Multiple pulses on the same channel are played **sequentially**. Pulses on separate channels are scheduled in parallel; i.e. **each pulse is played as soon as it's output channel is available**. 

## Two consecutive pulses on the same channel

In [ ]:
circuit = [
    #play two pulses on the Q0 drive channel
    {'name': 'pulse', 'phase': 0, 'freq': 4944383311, 'amp': 0.3, 'twidth': 2.4e-08,
     'env': {'env_func': 'cos_edge_square', 'paradict': {'ramp_fraction': 0.25}},
     'dest': 'Q0.qdrv'}, 
    
    {'name': 'pulse', 'phase': 0, 'freq': 4944383311, 'amp': 0.6, 'twidth': 2.4e-08,
     'env': {'env_func': 'cos_edge_square', 'paradict': {'ramp_fraction': 0.25}},
     'dest': 'Q0.qdrv'}
    
]

In [ ]:
compiled_prog = tc.run_compile_stage(circuit, fpga_config, None, compiler_flags={'resolve_gates': False})
asm_prog = tc.run_assemble_stage(compiled_prog, channel_configs)

In [ ]:
em.load_program(asm_prog)
result = em.execute()

In [ ]:
result.graph_channel('Q0.qdrv')

## Add a pulse on a different channel

In [ ]:
circuit = [
    # play two pulses on the Q0 drive channel
    {'name': 'pulse', 'phase': 0, 'freq': 4944383311, 'amp': 0.3, 'twidth': 2.4e-08,
     'env': {'env_func': 'cos_edge_square', 'paradict': {'ramp_fraction': 0.25}},
     'dest': 'Q0.qdrv'}, 
    
    {'name': 'pulse', 'phase': 0, 'freq': 4944383311, 'amp': 0.6, 'twidth': 2.4e-08,
     'env': {'env_func': 'cos_edge_square', 'paradict': {'ramp_fraction': 0.25}},
     'dest': 'Q0.qdrv'},
    
    # add a single pulse on Q1 drive -- this will be scheduled immediately, and played in parallel with the first Q0 pulse
    {'name': 'pulse', 'phase': 0, 'freq': 4944383311, 'amp': 0.6, 'twidth': 2.4e-08,
     'env': {'env_func': 'cos_edge_square', 'paradict': {'ramp_fraction': 0.25}},
     'dest': 'Q1.qdrv'}
    
]

In [ ]:
compiled_prog = tc.run_compile_stage(circuit, fpga_config, None, compiler_flags={'resolve_gates': False})
asm_prog = tc.run_assemble_stage(compiled_prog, channel_configs)

In [ ]:
em.load_program(asm_prog)
result = em.execute()

In [ ]:
result.graph_channel('Q0.qdrv')
result.graph_channel('Q1.qdrv')

### Timing Constraints

Delays and barriers can be used to control when pulses are played relative to one another:
  - `delay`: delay subsequent pulses on all channels in `scope` by amount `t` (in seconds). If `scope` is not provided, delay all channels
  - `barrier`: insert a scheduling barrier that applies to all channels in `scope`. All subsequent pulses on those channels are scheduled to play after the last pulse before the barrier (on those same channels) has finished (this is very similar to the behavior of a barrier in Qiskit). As with the `delay` instruction, if no channels are provided in `scope`, the barrier is applied to all channels.


In [ ]:
circuit = [
    #play two pulses on the Q0 drive channel
    {'name': 'pulse', 'phase': 0, 'freq': 4944383311, 'amp': 0.3, 'twidth': 2.4e-08,
     'env': {'env_func': 'cos_edge_square', 'paradict': {'ramp_fraction': 0.25}},
     'dest': 'Q0.qdrv'}, 
    
    {'name': 'delay', 't': 50.e-9},
    
    {'name': 'pulse', 'phase': 0, 'freq': 4944383311, 'amp': 0.6, 'twidth': 2.4e-08,
     'env': {'env_func': 'cos_edge_square', 'paradict': {'ramp_fraction': 0.25}},
     'dest': 'Q0.qdrv'},
    
    {'name': 'barrier'}, 
    
    {'name': 'pulse', 'phase': 0, 'freq': 4944383311, 'amp': 0.6, 'twidth': 2.4e-08,
     'env': {'env_func': 'cos_edge_square', 'paradict': {'ramp_fraction': 0.25}},
     'dest': 'Q1.qdrv'}
    
]

Compile, run, and plot the output below:

In [ ]:
compiled_prog = tc.run_compile_stage(circuit, fpga_config, None, compiler_flags={'resolve_gates': False})
asm_prog = tc.run_assemble_stage(compiled_prog, channel_configs)
em.load_program(asm_prog)
result = em.execute()
result.graph_channel('Q0.qdrv')
result.graph_channel('Q1.qdrv')

#### Extra: Scoped Delays and Barriers

Delays and barriers can be scoped to a subset of channels using the `scope` field (e.g. `{'name': 'barrier', 'scope': ['Q0', 'Q2']}`). 

## Exercise
Modify the pulse sequence below to have the following structure:

In [ ]:
# add barriers to the circuit below:

circuit = [
    {'name': 'pulse', 'phase': 0, 'freq': 4944383311, 'amp': 0.3, 'twidth': 2.4e-08,
     'env': {'env_func': 'cos_edge_square', 'paradict': {'ramp_fraction': 0.25}},
     'dest': 'Q0.qdrv'},
    
    {'name': 'pulse', 'phase': 0, 'freq': 4944383311, 'amp': 0.3, 'twidth': 2.4e-08,
     'env': {'env_func': 'cos_edge_square', 'paradict': {'ramp_fraction': 0.25}},
     'dest': 'Q2.qdrv'},
    
    {'name': 'barrier'},
    
    {'name': 'pulse', 'phase': 0, 'freq': 4944383311, 'amp': 0.3, 'twidth': 2.4e-08,
     'env': {'env_func': 'cos_edge_square', 'paradict': {'ramp_fraction': 0.25}},
     'dest': 'Q0.qdrv'}, 
    
    {'name': 'barrier'},

    
    {'name': 'pulse', 'phase': 0, 'freq': 4944383311, 'amp': 0.3, 'twidth': 2.4e-08,
     'env': {'env_func': 'cos_edge_square', 'paradict': {'ramp_fraction': 0.25}},
     'dest': 'Q1.qdrv'},

    {'name': 'barrier'},    
    
    {'name': 'pulse', 'phase': 0, 'freq': 4944383311, 'amp': 0.3, 'twidth': 2.4e-08,
     'env': {'env_func': 'cos_edge_square', 'paradict': {'ramp_fraction': 0.25}},
     'dest': 'Q2.qdrv'}, 
    
]
    
compiled_prog = tc.run_compile_stage(circuit, fpga_config, None, compiler_flags={'resolve_gates': False})
asm_prog = tc.run_assemble_stage(compiled_prog, channel_configs)

In [ ]:
em.load_program(asm_prog)
result = em.execute()
result.graph_channel('Q0.qdrv')
result.graph_channel('Q1.qdrv')

# Quantum Gate-level Programming

## Quantum logical operations

In general, a single-qubit state can be represented as an arbitrary superposition over the $|0 \rangle$ and $|1 \rangle$ states:

$$|\psi \rangle = \alpha|0\rangle + \beta|1\rangle $$

or equivalently as a vector:

$$|\psi \rangle = \begin{pmatrix} \alpha \\ \beta \end{pmatrix}$$

where $\alpha$ and $\beta$ are complex coefficients such that $|\alpha|^2 + |\beta|^2 = 1$. 

Multi-qubit states generalize this concept; an N-qubit state can be represented as a $2^N$ element vector. For example, a two qubit state can be represented as: 

$$|\psi \rangle = \alpha|00 \rangle + \beta|01 \rangle + \delta|10 \rangle + \gamma|11 \rangle$$

where $|\alpha|^2 + |\beta|^2 + |\delta|^2 + |\gamma|^2= 1$

**Quantum gates** are unitary matrix operators that manipulate the state. For example, the Pauli $X$ operator: $\sigma_X = \begin{pmatrix} 0 && 1 \\ 1 && 0 \end{pmatrix}$ maps $|\psi \rangle = \alpha|0\rangle + \beta|1\rangle $ to:

$$ \begin{pmatrix} 0&&1\\1&&0 \end{pmatrix} \begin{pmatrix} \alpha \\ \beta \end{pmatrix} = \begin{pmatrix} \beta \\ \alpha \end{pmatrix}$$

Gates can be generalized to multiple qubits; for example the CNOT (controlled-not) gate operates on the two-qubit state $|\psi \rangle = \alpha|00 \rangle + \beta|01 \rangle + \delta|10 \rangle + \gamma|11 \rangle$:

$$ CNOT |\psi \rangle = \begin{pmatrix} 1 && 0 && 0 && 0 \\ 0 && 1 && 0 && 0 \\ 0 && 0 && 0 && 1 \\ 0 && 0 && 1 && 0 \end{pmatrix} \begin{pmatrix} \alpha \\ \beta \\ \delta \\ \gamma \end{pmatrix} = \begin{pmatrix} \alpha \\ \beta \\ \gamma \\ \delta \end{pmatrix}$$

Similar to classical computation, any algorithm can be implemented by applying a sequence of gates drawn from a set of **universal** gates to a register of (qu)bits. An example of a universal classical gate set is $\{NAND\}$; analogously, an example universal quantum gate set is $\{X_{90}, CNOT, Z(\phi)\}$.

## Native gates

Any physical implementation of a quantum computer will implement a set of primitive gate operations, which together can be combined to implement any arbitrary computation. This set is referred to as that quantum computer's **native gate set**. In addition to pulse-level programming, QubiC also supports programming at the native gate level.

Each native gate consists of a set of pulses that implement the desired unitary (logical) operation on a particular qubit. For example, the "Q0X90" gate implements a 90-degree rotation about the Pauli $X$-axis on qubit "Q0". This mapping between native gates and pulses is determined using a calibration procedure, and for QubiC, is stored in a json file (example: qubitcfg.json) and referenced in QubiC circuits. 

## Example of X90 gate on Q0

## Example of CNOT gate, consisting of a composite list of pulses

In [ ]:
circuit = [
    # this circuit plays calibrated X90 gates
    {'name': 'X90', 'qubit': 'Q0'},
    {'name': 'X90', 'qubit': 'Q1'}]

In [ ]:
# load in configuration from qubitcfg.json
qchip = qc.QChip('./config/qubitcfg.json')

In [ ]:
# inspect a gate
qchip.gates['Q0X90'].cfg_dict

In [ ]:
# link the qchip configuration at compile time
compiled_prog = tc.run_compile_stage(circuit, fpga_config, qchip)
asm_prog = tc.run_assemble_stage(compiled_prog, channel_configs)

In [ ]:
em.load_program(asm_prog)
result = em.execute()
result.graph_channel('Q0.qdrv')
result.graph_channel('Q1.qdrv')

# Single-qubit Rotations Using QuTiP

Quantum states can be visualized as points (or vectors) on the surface of a sphere (known as the **Bloch sphere**) using the following parameterization:

$$ |\psi \rangle = \cos{\frac{\theta}{2}} |0 \rangle + \sin{\frac{\theta}{2}} e^{i\phi} |1 \rangle$$

<center><img src="images/Bloch_sphere.svg"/></center>

QuTiP (Quantum Toolbox in Python) is an open-source software package that is used for simulating the evolution of quantum states over time. For a full documentation of QuTiP, visit this [link](https://qutip.org/documentation.html).

In our emulator, the user can sequence drive pulses to the qubit in the .qdrv channels. These pulses are directly applied to the qubit, and drive the quantum state of the qubit. Within the `qubitcfg.json` file there are defined gates that have parameters that will drive the qubit as intended. For example, the parameters for the Q0 X90 gate are specific for that qubit to rotate the state vector 90 degrees around the X axis. Using QuTiP's `mesolve()` function, the emulator is able to calculate the evolution of the qubit using the drive Hamiltonian mentioned earlier.

## Single $X_{90}$ Gate

In [ ]:
prog = [{'name': 'X90', 'qubit': 'Q0'}]
compiled_prog = tc.run_compile_stage(prog, fpga_config=FPGAConfig(), qchip=qchip)
binary = tc.run_assemble_stage(compiled_prog, channel_configs)
em.load_program(binary)
result = em.execute(toggle_qubits=True)

In [ ]:
result.graph_bloch('Q0')

The X90 gate implements a 90 degree rotation about the X-axis of the Bloch sphere. The qubit is initialized in state $|0 \rangle$ (shown by the green dot). The state evolves to its final position during the application of the RF pulse (green arrow).

### Amplitude Modification

The amplitude of our calibrated $X_{90}$ pulse is 0.1 (in DAC full scale units). Let's try changing the amplitude and see what happens to the quantum state:

In [ ]:
new_amp = 0.22

prog = [{'name': 'X90', 'qubit': 'Q0', 'modi': {(0, 'amp'): new_amp}}]
# The 'modi' key allows us to modify gate parameters in-place for a single gate/pulse

compiled_prog = tc.run_compile_stage(prog, fpga_config=FPGAConfig(), qchip=qchip)
binary = tc.run_assemble_stage(compiled_prog, channel_configs)
em.load_program(binary)
result = em.execute(toggle_qubits=True)

In [ ]:
result.graph_bloch('Q0')

## (Virtual) $Z$ Gates

Arbitrary rotations of the quantum state about the $Z$ axis of the Bloch sphere can be performed *virtually* in software, by incrementing the relative phase of subsequent pulses. This effectively **rotates** the $x$ and $y$ axes relative to the qubit state vector, performing a coordinate transformation.

In [ ]:
# try modifying the relative phase of the second pulse
relative_phase = 3*np.pi/4

prog = [{'name': 'X90', 'qubit': 'Q0'},
        {'name': 'X90', 'qubit': 'Q0', 'modi': {(0, 'phase'): relative_phase}}]

In [ ]:
compiled_prog = tc.run_compile_stage(prog, fpga_config=FPGAConfig(), qchip=qchip)
binary = tc.run_assemble_stage(compiled_prog, channel_configs)
em.load_program(binary)
result = em.execute(toggle_qubits=True)
result.graph_bloch('Q0')

In [ ]:
# we can also do this with a virtual-Z instruction
relative_phase = 3*np.pi/4

prog = [{'name': 'X90', 'qubit': 'Q0'},
        {'name': 'virtual_z', 'qubit': 'Q0', 'phase': relative_phase},
        {'name': 'X90', 'qubit': 'Q0'}]

In [ ]:
compiled_prog = tc.run_compile_stage(prog, fpga_config=FPGAConfig(), qchip=qchip)
binary = tc.run_assemble_stage(compiled_prog, channel_configs)
em.load_program(binary)
result = em.execute(toggle_qubits=True)
result.graph_bloch('Q0')

# Qubit State Measurement

For a qubit with state $|\psi \rangle = \alpha|0\rangle + \beta|1\rangle $, we don't have direct access to the amplitudes $\alpha$, $\beta$. Instead, we can perform a measurement on the qubit, and obtain $|0\rangle$ with probability $\alpha^2$ and $|1 \rangle$ with probability $\beta^2$. 

This concept generalizes to multi-qubit states; for instance measuring the two-qubit state $|\psi \rangle = \alpha|00 \rangle + \beta|01 \rangle + \delta|10 \rangle + \gamma|11 \rangle$ will have possible outcomes 00, 01, 10, or 11 with probabilities $\alpha^2$, $\beta^2$, $\delta^2$, or $\gamma^2$, respectively.

## Superconducting Qubit Readout

For superconducting qubits, each qubit is coupled to a **readout resonator** circuit (which is on the same substrate as the qubit). To measure the qubit, we drive the resonator with an on-resonance pulse; the qubit state will modulate the resonator frequency, which we can measure in the phase response of the resonator to our drive pulse. 

In the control system, the qubit readout signal chain has two phases:
 1. Using the readout DAC to play a **drive pulse** that excites the readout resonator
 2. Recording the response using the **readout ADC**, and measuring the the change in resonator frequency (hence qubit state) using **digital IQ downconversion**
 
## Readout Drive

The readout drive signal path looks like this:

![image](images/dac_signal_path.drawio.svg)

As you might have guessed from the diagram, the readout chain is **frequency multiplexed**. That is, a bank of **8 independent firmware signal generators** can be used to drive the readout DAC, which allows **8 qubits to be read out simultaneously** (provided their readout resonators are coupled to a common bus). Readout pulses can be scheduled and parameterized like any other pulse.

**Note**: As with the previous demo we don't have a physical DAC, so the readout drive maps to `dac_out[8]`

### Play a pulse on the readout drive channel

Pulses can be played on readout drive channels by setting the `dest` field to `Qn.rdrv`. In the channel configuration, this corresponds to a `core_ind` that is the same as the drive channel for that qubit, and `elem_ind = 1`.

In [ ]:
circuit = [{'name': 'pulse', 'phase': 0, 'freq': 6.1e9, 'amp': 0.6, 'twidth': 1e-07,
     'env': {'env_func': 'cos_edge_square', 'paradict': {'ramp_fraction': 0.25}},
     'dest': 'Q0.rdrv'}]

In [ ]:
compiled_prog = tc.run_compile_stage(circuit, fpga_config, None, compiler_flags={'resolve_gates': False})
asm_prog = tc.run_assemble_stage(compiled_prog, channel_configs)

In [ ]:
em.load_program(asm_prog)
result = em.execute()
result.graph_dac('DAC0')

Now try it with multiple readout pulses on different qubits. Note that readout pulses are arbitrary/can be optimized per qubit.

In [ ]:
circuit = [{'name': 'pulse', 'phase': 0, 'freq': 6.1e9, 'amp': 0.4, 'twidth': 1e-07,
     'env': {'env_func': 'cos_edge_square', 'paradict': {'ramp_fraction': 0.25}},
     'dest': 'Q0.rdrv'},
           {'name': 'pulse', 'phase': 0, 'freq': 6.2e9, 'amp': 0.4, 'twidth': 1e-07,
     'env': {'env_func': 'cos_edge_square', 'paradict': {'ramp_fraction': 0.25}},
     'dest': 'Q1.rdrv'}]

In [ ]:
## compile, assemble and plot here. When plotting, try zooming in to a small time range to see the different frequency components
compiled_prog = tc.run_compile_stage(circuit, fpga_config, None, compiler_flags={'resolve_gates': False})
asm_prog = tc.run_assemble_stage(compiled_prog, channel_configs)

em.load_program(asm_prog)
result = em.execute()
result.graph_dac('DAC0')

In [ ]:
# If we plot an FFT of the above signal we can see both frequency components:
rdrv_dac_data = result.get_dac_data('DAC0')['voltage']
fftfreq = np.fft.rfftfreq(len(rdrv_dac_data), 0.125e-9)
rdrv_fft = np.fft.rfft(rdrv_dac_data)
plt.plot(fftfreq, np.abs(rdrv_fft))
plt.xlabel('freq (Hz)')


**Note**: The peaks we see here are the *image frequencies* in the first nyquist zone (0 - 4 GHz), reflected across the nyquist frequency of 4 GHz (`f_image = f_nyquist - (f - f_nyquist)`). With appropriate configuration of DACs and choice of filter, these images can be attenuated, leaving the tones in the second nyquist zone (4 - 8 GHz).

## Readout Demodulation

The firmware also has **8 signal generator channels** for generating **readout demodulation tones** (these are called **readout_demod** or **rdlo** channels, for readout local-oscillator). These demodulation tones are **mixed with the signal from the readout ADC** to downconvert that readout tone to baseband. The full readout demodulation signal path looks like this:

![image](images/readout_chain.drawio.svg)

Note the accumulators after the mixing stage. Any pulse on a readout_demod (rdlo) channel will **reset and trigger the corresponding accumulator**, which will integrate the mixed signal for the duration of the rdlo pulse.

Below, we generate a pulse on the `rdlo` channel in addition to the `rdrv` channel. Both of these pulses have the same frequency as the simulated `Q0` readout resonator frequency:

In [ ]:
prog = [{'name': 'pulse', 'phase': 0, 'freq': 6.55e9, 'amp': 1, 'twidth': 1e-6,
                'env': {'env_func': 'cos_edge_square', 'paradict': {'ramp_fraction': 0.15}},
                'dest': 'Q0.rdrv'},
        {'name': 'pulse', 'phase': 0, 'freq': 6.55e9, 'amp': 1, 'twidth': 1e-6,
                'env': {'env_func': 'square', 'paradict': {}},
                'phase': 0.89*np.pi, 'dest': 'Q0.rdlo'}]

# Compile and assemble the circuit to get the binaries to feed our emulator
compiled_prog = tc.run_compile_stage(prog, fpga_config=FPGAConfig(), qchip=qchip)
binary = tc.run_assemble_stage(compiled_prog, channel_configs)

# Load and execute simulation
em.load_program(binary)
result = em.execute(toggle_resonator=True, toggle_qubits=True)
result.graph_dac('DAC0') # rdrv channels belong to 0th dac

We can see the simulated resonator response to our drive pulse in the ADC signal:

In [ ]:
result.graph_adc('ADC0')

Check the accumulated value:

In [ ]:
zero_result = result.iq_values
zero_result

In the real hardware, these values are saved to a memory buffer that can be read by the host PC (size is configurable, current builds support up to 1024 shots per channel).

As with control pulses, we can use a calibrated `read` "gate" to 

We can now prepare `Q0` in the $|1 \rangle$ state, and observe the changes in the resonator response and integrated/accumulated value:

In [ ]:
prog = [
    {'name': 'X90', 'qubit': 'Q0'},
    {'name': 'X90', 'qubit': 'Q0'},
    {'name': 'read', 'qubit': 'Q0'}]

# Compile and assemble the circuit to get the binaries to feed our emulator
compiled_prog = tc.run_compile_stage(prog, fpga_config=FPGAConfig(), qchip=qchip)
binary = tc.run_assemble_stage(compiled_prog, channel_configs)

# Load and execute simulation
em.load_program(binary)
result = em.execute(toggle_resonator=True, toggle_qubits=True)
result.graph_adc('ADC0')

In [ ]:
one_result = result.iq_values
one_result

In [ ]:
plt.plot(zero_result.real, zero_result.imag, '.', label='zero')
plt.plot(one_result.real, one_result.imag, '.', label='one')
plt.xlabel('I')
plt.ylabel('Q')

Now try applying a single $X_{90}$:

In [ ]:
n_shots = 100

n_ones = 0
for i in range(n_shots):
    prog = [
    {'name': 'X90', 'qubit': 'Q0'},
    {'name': 'barrier'},
    {'name': 'pulse', 'phase': 0, 'freq': 6553826000.000857, 'amp': 1, 'twidth': 1e-6,
                'env': {'env_func': 'cos_edge_square', 'paradict': {'ramp_fraction': 0.15}},
                'dest': 'Q0.rdrv'},
    {'name': 'pulse', 'phase': 0, 'freq': 6553826000.000857, 'amp': 1, 'twidth': 1e-6,
                'env': {'env_func': 'cos_edge_square', 'paradict': {'ramp_fraction': 0.15}},
                'phase': np.pi/2, 'dest': 'Q0.rdlo'}]

    # Compile and assemble the circuit to get the binaries to feed our emulator
    compiled_prog = tc.run_compile_stage(prog, fpga_config=FPGAConfig(), qchip=qchip)
    binary = tc.run_assemble_stage(compiled_prog, channel_configs)
    
    # Load and execute simulation
    em.load_program(binary)
    result = em.execute(tags=['FPROC'], toggle_resonator=True, toggle_qubits=True)
    n_ones += result.fproc[0]
    print(result.iq_values)

In [ ]:
print(f'zeros population: {1-n_ones/n_shots}; ones population: {n_ones/n_shots}')

# Branching/Feedforward using Mid-circuit Measurements 

QubiC is capable of making arbitrary real-time control decisions based on measurement results. The latency of these operations (~200-300 ns) is well within the coherence time of superconducting transmon qubits (most of this latency comes from pulse generation/mixing, not branching logic). The distributed processor cores make branching decisions by interacting with a firmware module called the "function processor" (FPROC):

![image](images/qubic_arch.png)

Depending on the implementation, the FPROC can **aggregate measurement results**, or optionally do some **application specific data processing**. Each distributed processor **core** can **request/receive results** from the FPROC using a special instruction. In the current implementation for this simulation, the FPROC simply aggregates thresholded measurement results.

## `branch_fproc` Instruction

The instruction used for **branching** looks like this:

```json
{'name': 'branch_fproc', 'alu_cond': <'le' or 'ge' or 'eq'>, 'cond_lhs': <var or ival>, 
'func_id': function_id, 'scope': <list_of_qubits_or_channels> 'true': [instruction_list], 'false': [instruction_list]}
```

Let's break this down:
 1. Data is **requested** from the FPROC according to the provided `func_id`. In the version of the gateware we're simulating, `func_id` just indicates the qubit whose measurement result we want. A list of available FPROC channels can be found in `fpga_config.fproc_channels`
 2. Once the FPROC receives the request, it fetches the result of the **most recent previous measurement** on that channel and sends it to the **core(s)** that requested it
 3. The **core** makes a **branching decision** according to `cond_lhs <alu_cond> fproc_result`. For example, you can check if the measurement was 0 using: `cond_lhs = 0`, `alu_cond = 'eq'`, which implements: `0 == fproc_result`.
 4. If the expression evaluates to **True**, the block of instructions in the `true` field are executed, **else** the block in `false` is executed.
 
 ### Scheduling
 
If there are multiple branching statements in sequence, e.g. 
```json
{'name': 'branch_fproc', 'alu_cond': 'eq', 'cond_lhs': 0, 
'func_id': 'Q0.meas', 'scope': ['Q1', 'Q2'],
     'true': [
         {'name': 'X90', 'qubit': 'Q1'}
        ], 
    'false': [
        {'name': 'X90', 'qubit': 'Q1'}
    ]
},

{'name': 'branch_fproc', 'alu_cond': 'eq', 'cond_lhs': 0, 
'func_id': 'Q0.meas', 'scope': ['Q3'], 
     'true': [
         {'name': 'X90', 'qubit': 'Q3'}
        ], 
    'false': [
    ]
}
```

they are executed **concurrently**, as long as the conditional instructions (i.e. scope) are on different sets of qubits.


## A note about state classification/thresholding

In the standard QubiC gateware, the FPROC classsifies states by thresholding across the y-axis; any accumulated value with `x>0` gets classified to 0; `x<0` goes to 1. QubiC also supports a proprietary ML backend that uses an on-FPGA machine-learning based state discriminator which supports qudit states -- this backend will be presented later in the tutorial.

In this simulation wavefunction collapse and state classification is handled by QuTiP (since we have direct access to the full qubit state).

![image](images/reset_image.svg)

## FPROC Channels

A list of available FPROC channels in the current gateware can be found in `fpga_config.fproc_channels`:

In [ ]:
fpga_config.fproc_channels

Each named channel indexes a `FPROCChannel` object, which contains the physical channel `id` to use, as well as the additional `delay` required to access the measurement (i.e. the additional delay, after the measurement pulse, required before the pulse is available).

The channel `id` is resolved by the assembler using the `channel_configs` object; e.g. for `Q0.meas`, the physical channel ID is given by `channel_configs['Q0.rdlo'].core_ind`.

The delay is used by the scheduler to ensure that all subsequent pulses are scheduled **after** the measurement becomes available.

## Circuit with Branching

Branch statements are a valid instruction type, and can be included in circuits

In [ ]:
circuit = [
    {'name': 'X90', 'qubit': 'Q0'},
    {'name': 'read', 'qubit': 'Q0'}, #use a read gate, which includes an rdrv pulse followed by an rdlo pulse
    {'name': 'branch_fproc', 'alu_cond': 'eq', 'cond_lhs': 1, 'func_id': 'Q0.meas', 'scope': ['Q0'], 
     'true': [
         {'name': 'X90', 'qubit':'Q0'}], 
     'false': []},
    {'name': 'read', 'qubit': 'Q0'}]


In [ ]:
compiled_prog = tc.run_compile_stage(circuit, fpga_config, qchip)
asm_prog = tc.run_assemble_stage(compiled_prog, channel_configs)

The compiler lowers **branch statements** to **conditional jumps** that execute on the distributed processor core:

In [ ]:
compiled_prog.program

In [ ]:
# run the program, and check the acc result
em.load_program(asm_prog)
result = em.execute(tags=['FPROC'], toggle_qubits=True, toggle_resonator=True)
result.graph_channel('Q0.qdrv')

The accumulated result has a real part > 0, so we expect no pulses on the qdrv channel:

# Other IR Instructions

## `declare` Instruction

```json
{'name': 'declare', 'var': <var_name>, 'scope': <list_of_qubits_or_channels>, 'type': <'phase' or 'amp' or 'int'>}
```
Declares a variable `var_name`. The `scope` is a list of qubits or channels that might reference the variable; for example if the variable was a loop index over qubit drive operations on Q1 and Q0, you could set `scope: ['Q0', 'Q1']` or `scope: ['Q0.qdrv', 'Q1.qdrv']`. `type` is optional and defaults to `'int'`, which is used for general purpose operations. `'phase'` and `'amp'` are types used for register-based pulse parameterization.

## `set_var` Instruction

```json
{'name': 'set_var', 'value': <value>, 'var': <var_name>}
```
This instruction simply sets the value of `var_name` to `value`

## `alu_op` Instruction

```json
{'name': 'alu', 'op': 'add' or 'sub' or 'le' or 'ge' or 'eq', 'lhs': var_name or value, 'rhs': var_name, 'out': output reg}
```

Performs a binary operation and stores it in a variable. The inputs to the operation are given by `lhs` and `rhs`. `lhs` can be an immediate value or register, and `rhs` must be a variable. Result is stored in `out` (can be the same as `lhs` and/or `rhs`).


## `branch_var` Instruction

Same behavior as `branch_fproc`, except a variable is used in place of a measurement result.

```json
{'name': 'branch_var', 'alu_cond': <'le' or 'ge' or 'eq'>, 'cond_lhs': <var or ival>, 
 'cond_rhs': <var>, 'scope': <list_of_qubits_or_channels> 'true': [instruction_list], 'false': [instruction_list]}
```

## `loop` Instruction

The loop instruction can be used to repeat pulse sequences in hardware, with minimal latency:

```json
{'name': 'loop', 'cond_lhs': <reg or ival>, 'alu_cond': <'ge', 'le', 'eq'>, 'cond_rhs': var_name, 'scope': <list_of_qubits>, 'body': [instruction_list]}
```
This instruction will execute the instruction list in `'body'` while the condition specified in the loop instruction (`cond_lhs <alu_cond> cond_rhs)` evaluates to true. To use loops effectively, we also introduce instructions that declare and operate on **variables**:

In [ ]:
# repeat a Q0 drive pulse 10 times:
circuit = [
    {'name': 'declare', 'var': 'loop_ind', 'scope': ['Q0']},
    {'name': 'set_var', 'value': 0, 'var': 'loop_ind'},
    #{'name': 'delay', 't': 10.e-9},
    {'name': 'loop', 'cond_lhs': 10, 'alu_cond': 'ge', 'cond_rhs': 'loop_ind', 'scope': ['Q0'], 
     'body': [
         {'name': 'X90', 'qubit': 'Q0'},
         {'name': 'alu', 'op': 'add', 'lhs': 1, 'rhs': 'loop_ind', 'out': 'loop_ind'}
     ]}]

In [ ]:
compiled_prog = tc.run_compile_stage(circuit, fpga_config, qchip)
asm_prog = tc.run_assemble_stage(compiled_prog, channel_configs)
compiled_prog.program

In [ ]:
em.load_program(asm_prog)
result = em.execute()

In [ ]:
result.graph_channel('Q0.qdrv')

## Example: Fast Amplitude Sweep

Loops can also be used to implement parameter sweeps, using variables to parameterize pulses.

In [ ]:
# sweep Q0.drive amplitude:
circuit = [
    {'name': 'declare', 'var': 'loop_ind', 'scope': ['Q0']},
    {'name': 'set_var', 'value': 0, 'var': 'loop_ind'},
    {'name': 'declare', 'var': 'amp', 'scope': ['Q0'], 'dtype': 'amp'},
    {'name': 'set_var', 'value': 0.1, 'var': 'amp'}, # pulse amplitude is parameterized by processor register
    {'name': 'loop', 'cond_lhs': 10, 'alu_cond': 'ge', 'cond_rhs': 'loop_ind', 'scope': ['Q0'], 
     'body': [
            {'name': 'pulse', 'phase': 0, 'freq': 4944383311, 'amp': 'amp', 'twidth': 2.4e-08,
             'env': {'env_func': 'cos_edge_square', 'paradict': {'ramp_fraction': 0.25}},
             'dest': 'Q0.qdrv'},
         {'name': 'alu', 'op': 'add', 'lhs': 1, 'rhs': 'loop_ind', 'out': 'loop_ind'},
         {'name': 'alu', 'op': 'add', 'lhs': 0.1, 'rhs': 'amp', 'out': 'amp'}

         
     ]}]

In [ ]:
compiled_prog = tc.run_compile_stage(circuit, fpga_config, qchip)
asm_prog = tc.run_assemble_stage(compiled_prog, channel_configs)
compiled_prog.program

In [ ]:
em.load_program(asm_prog)
result = em.execute()
result.graph_channel('Q0.qdrv')

## Example: Fast Frequency Sweep

In [ ]:
# Need to declare all frequencies at compile-time, since they are stored as pre-computed phase offsets in memory, and accessed by address
freqs = np.linspace(0, 10.e6, 11) 
circuit = [{'name': 'declare_freq', 'freq': freq, 'scope': ['Q0.qdrv'], 'freq_ind': i} 
           for i, freq in enumerate(freqs)]

circuit.extend([
    {'name': 'declare', 'var': 'freq_ind', 'scope': ['Q0'], 'dtype': 'int'}, # parameterize the frequency using an index 
    {'name': 'set_var', 'value': 0, 'var': 'freq_ind'},
    {'name': 'loop', 'cond_lhs': len(freqs), 'alu_cond': 'ge', 'cond_rhs': 'freq_ind', 'scope': ['Q0'], 
              'body': [

                    {'name': 'delay', 't': 30.e-8},
                  
                    {'name': 'pulse', 'phase': 0, 'freq': 'freq_ind', 'amp': 0.9, 'twidth': 100e-08,
                         'env': {'env_func': 'cos_edge_square', 'paradict': {'ramp_fraction': 0.25}},
                         'dest': 'Q0.qdrv'},
                  
                    #{'name': 'read', 'qubit': 'Q0'},
                
                    {'name': 'alu', 'op': 'add', 'lhs': 1, 'rhs': 'freq_ind', 'out': 'freq_ind'},

                  
            ]}])

In [ ]:
compiled_prog = tc.run_compile_stage(circuit, fpga_config, qchip)
asm_prog = tc.run_assemble_stage(compiled_prog, channel_configs)

In [ ]:
em.load_program(asm_prog)
result = em.execute()
result.graph_channel('Q0.qdrv')

## Example: Compound Freq x Amplitude Sweep

In [ ]:
amp_start = 0.1
amp_stop = 0.5
amp_step = 0.1

freqs = np.linspace(0, 10.e6, 5) # frequencies can be an arbitrary array, 
                                          # since we're storing them in a table and accessing the address

# first, declare all frequencies
circuit = [{'name': 'declare_freq', 'freq': freq, 'scope': ['Q0.qdrv'], 'freq_ind': i} 
           for i, freq in enumerate(freqs)]

# fill in the rest of the circuit
circuit.extend([
    {'name': 'declare', 'var': 'amp', 'scope': ['Q0'], 'dtype': 'amp'},
    {'name': 'set_var', 'value': amp_start, 'var': 'amp'}, # pulse amplitude is parameterized by processor register
    
    {'name': 'declare', 'var': 'freq_ind', 'scope': ['Q0'], 'dtype': 'int'}, # frequency index is parameterized by a register
    
    # outer loop over amplitude
    {'name': 'loop', 'cond_lhs': amp_stop, 'alu_cond': 'ge', 'cond_rhs': 'amp', 'scope': ['Q0'], 
     'body': [
         {'name': 'set_var', 'value': 0, 'var': 'freq_ind'},
         #{'name': 'alu', 'op': 'add', 'lhs': 2, 'rhs': 'freq_ind', 'out': 'freq_ind'},

         # inner loop over frequency
         {'name': 'loop', 'cond_lhs': len(freqs), 'alu_cond': 'ge', 'cond_rhs': 'freq_ind', 'scope': ['Q0'], 
              'body': [

                    {'name': 'delay', 't': 100.e-9},
                  
                    {'name': 'pulse', 'phase': 0, 'freq': 'freq_ind', 'amp': 'amp', 'twidth': 50e-08,
                         'env': {'env_func': 'cos_edge_square', 'paradict': {'ramp_fraction': 0.25}},
                         'dest': 'Q0.qdrv'},
                  
                    #{'name': 'read', 'qubit': 'Q0'},
                
                    {'name': 'alu', 'op': 'add', 'lhs': 1, 'rhs': 'freq_ind', 'out': 'freq_ind'},

                  
            ]},
         {'name': 'alu', 'op': 'add', 'lhs': amp_step, 'rhs': 'amp', 'out': 'amp'},

        ]
    }
])

In [ ]:
compiled_prog = tc.run_compile_stage(circuit, fpga_config, qchip)
asm_prog = tc.run_assemble_stage(compiled_prog, channel_configs)
#compiled_prog.program

In [ ]:
em.load_program(asm_prog)
result = em.execute()
result.graph_channel('Q0.qdrv')

## Exercise: Combine Branching and Looping

Play 5 pulses on `Q0.qdrv`, each followed by a read. If the measurement result is 1, increment the amplitude by 0.1.

In [ ]:
circuit = [
    # initialize variables:
    {'name': 'declare', 'var': 'loop_ind', 'scope': ['Q0']},
    {'name': 'set_var', 'value': 0, 'var': 'loop_ind'},
    {'name': 'declare', 'var': 'amp', 'scope': ['Q0'], 'dtype': 'amp'},
    {'name': 'set_var', 'value': 0.1, 'var': 'amp'}, # pulse amplitude is parameterized by processor register
    
    {'name': 'loop', 'cond_lhs': 5, 'alu_cond': 'ge', 'cond_rhs': 'loop_ind', 'scope': ['Q0'], 
     'body': [
         
         # read (rdlo pulse):
        {'name': 'read', 'qubit': 'Q0'},
         
        
         # Fill in branch_fproc + conditional amplitude increment here:
         
         
         # pulse:
         {'name': 'pulse', 'phase': 0, 'freq': 4460029188.07884, 'amp': 'amp', 'twidth': 2.4e-08,
           'env': {'env_func': 'cos_edge_square', 'paradict': {'ramp_fraction': 0.25}},
           'dest': 'Q0.qdrv'},
         
         # increment loop counter
         {'name': 'alu', 'op': 'add', 'lhs': 1, 'rhs': 'loop_ind', 'out': 'loop_ind'}
     ]

         
     }]

In [ ]:
compiled_prog = tc.run_compile_stage(circuit, fpga_config, qchip)
asm_prog = tc.run_assemble_stage(compiled_prog, channel_configs)
compiled_prog.program

In [ ]:
em.load_program(asm_prog)
result = em.execute(toggle_qubits=True, toggle_resonator=True)
result.graph_channel('Q0.qdrv')